# Clean incident-retrieval model selection

This notebook compares BM25, MiniLM, and Mixedbread using the clean agent's frozen retrieval contract. It does not need Kubernetes, Redis, n8n, the dashboard, or an API key.

## What is being tested

`evidence -> stable EvidenceTemplate -> fingerprint exact check -> BM25 candidates -> optional local reranker`

Each stored example contains evidence plus approved hints. Exact fingerprints bypass reranking; all other cases produce one of `exact / nearest / none / degraded`. MiniLM and Mixedbread receive the same BM25 candidate set.

The LLM judge is intentionally excluded: it is outside the local retrieval comparison and adds provider cost, latency, and non-determinism.


## 1. Get the clean code and lightweight dependency

Run this once in a new Colab runtime. Model-specific dependencies are installed in a later section.


In [ ]:
!git clone https://github.com/Ama2352/vroom-services.git
%cd /content/vroom-services/incident-diagnosis/evaluation
!pip -q install rank-bm25

## 2. Load the frozen cases

The snapshot uses the clean `families`, `examples`, and `hints` schema. No production data is read or changed.


In [ ]:
from pathlib import Path
import sys

EVALUATION_ROOT = Path.cwd()
AGENT_ROOT = EVALUATION_ROOT.parent / 'agent'
sys.path[:0] = [str(AGENT_ROOT), str(EVALUATION_ROOT)]

from benchmark import load_cases, load_snapshot

cases = load_cases(EVALUATION_ROOT / 'fixtures' / 'retrieval_cases.json')
snapshot = load_snapshot(EVALUATION_ROOT / 'fixtures' / 'knowledge_snapshot.json')
print(f'{len(cases)} frozen cases; {len(snapshot["examples"])} approved examples')

## 3. Download pinned local rerankers

The download is revision-pinned and the ONNX artifact is checksum-verified. These models stay local to the Colab runtime.


In [ ]:
!pip -q install transformers onnxruntime huggingface_hub numpy matplotlib psutil

from huggingface_hub import snapshot_download
from retrieval.reranker import MiniLMReranker, ModelSpec, OnnxCrossEncoder, verify_sha256
from benchmark import IdentityReranker, load_model_specs, retrieve_case, run_system, passes_gate

model_specs = load_model_specs(EVALUATION_ROOT / 'model_specs.json')

def download_model(name):
    spec = model_specs[name]
    model_dir = Path(snapshot_download(
        repo_id=spec['repo_id'], revision=spec['revision'],
        allow_patterns=['*.json', '*.txt', '*.model', 'onnx/*.onnx'],
    ))
    verify_sha256(model_dir / spec['onnx_file'], spec['sha256'])
    return model_dir, spec

def build_reranker(model_dir, spec):
    backend = OnnxCrossEncoder(model_dir, ModelSpec(**spec))
    return MiniLMReranker(backend)


## 4. Run the same pipeline for every system

BM25 selects candidates first. The two neural models can only reorder that same small candidate set. Exact fingerprint matches bypass every reranker.


In [ ]:
import time
import numpy as np
import psutil

def measure_local_system(name):
    model_dir, spec = download_model(name)
    process = psutil.Process()
    started = time.perf_counter()
    reranker = build_reranker(model_dir, spec)
    cold_load_ms = (time.perf_counter() - started) * 1000

    latencies = []
    for case in cases:
        started = time.perf_counter()
        retrieve_case(case, snapshot, reranker)
        latencies.append((time.perf_counter() - started) * 1000)

    artifact = model_dir / spec['onnx_file']
    return run_system(cases, snapshot, reranker, name=name), {
        'artifact_mb': artifact.stat().st_size / 1024**2,
        'cold_load_ms': cold_load_ms,
        'p50_ms': float(np.percentile(latencies, 50)),
        'p95_ms': float(np.percentile(latencies, 95)),
        'peak_rss_mb': process.memory_info().rss / 1024**2,
    }

results = {'bm25': run_system(cases, snapshot, IdentityReranker(), name='bm25')}
operations = {'bm25': {'artifact_mb': 0.0, 'cold_load_ms': 0.0, 'p50_ms': 0.0, 'p95_ms': 0.0, 'peak_rss_mb': 0.0}}
unavailable = {}

for name in ('minilm', 'mixedbread_xsmall'):
    try:
        results[name], operations[name] = measure_local_system(name)
    except Exception as exc:
        unavailable[name] = f'{type(exc).__name__}: {exc}'

print('Unavailable systems:', unavailable or 'none')

## 5. Quality metrics

Exact correctness and advisory ranking quality are shown separately. A confusion matrix is intentionally omitted because this is a retrieval-ranking comparison with multiple valid candidates and abstentions, not a single-label classifier.


In [ ]:
import matplotlib.pyplot as plt

def ratio(numerator, denominator):
    return numerator / denominator if denominator else 0.0

names = list(results)
quality = {
    name: [
        ratio(result.exact_correct, result.exact_total),
        ratio(result.advisory_top1, result.advisory_positive_count),
        ratio(result.advisory_recall_at_3, result.advisory_positive_count),
        ratio(result.advisory_mrr_sum, result.advisory_positive_count),
    ]
    for name, result in results.items()
}

labels = ['Exact correctness', 'Advisory Top-1', 'Recall@3', 'MRR']
x = np.arange(len(labels))
width = 0.8 / len(names)
fig, ax = plt.subplots(figsize=(10, 4))
for index, name in enumerate(names):
    ax.bar(x - 0.4 + width / 2 + index * width, quality[name], width, label=name)
ax.set_xticks(x, labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Quality metrics')
ax.legend()
plt.show()

## 6. Safety gate and Operational metrics

A model must have no forbidden result, false positive, exact failure, or degraded result. It must also match or exceed BM25 ranking quality, stay below 1,000 ms p95 latency, and use at most 512 MB RSS.


In [ ]:
baseline = results['bm25']
eligibility = {}
for name, result in results.items():
    safety_ok = passes_gate(result, baseline=baseline)
    operations_ok = operations[name]['p95_ms'] <= 1000 and operations[name]['peak_rss_mb'] <= 512
    eligibility[name] = safety_ok and operations_ok
    print(f'{name:18} false positives={result.false_positives}  forbidden={result.forbidden_acceptances}  exact failures={result.exact_failures}  abstentions={result.correct_abstentions}  pass={eligibility[name]}')

metrics = ['artifact_mb', 'cold_load_ms', 'p95_ms', 'peak_rss_mb']
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for axis, metric in zip(axes, metrics):
    axis.bar(names, [operations[name][metric] for name in names])
    axis.set_title(metric.replace('_', ' '))
    axis.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

candidates = [name for name in names if name != 'bm25' and eligibility[name]]
if candidates:
    winner = min(candidates, key=lambda name: (operations[name]['p95_ms'], operations[name]['artifact_mb']))
    print(f'Decision: choose {winner}; it passed the quality, safety, latency, and memory gates.')
else:
    print('Decision: keep BM25-only for now; no local reranker passed every gate.')